# 00 — Setup and sanity check

Run this first in any new Colab session to confirm the data loaded correctly before doing real work in another notebook. Every notebook in this repo starts with the same bootstrap cell below.

In [ ]:
import os

from google.colab import userdata

token = userdata.get("GH_PAT")  # set once via the key icon in Colab's left sidebar

BRANCH = "issue-1-meds-reader-wrapper"  # change this back to "main" once PR #21 is merged

REPO_DIR = "ehrshot-patient-graphs"
if os.path.exists(REPO_DIR):
    # Re-running without a fresh VM (no restart) — make sure we're actually
    # on BRANCH's latest commit instead of silently keeping a stale clone.
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} https://{token}@github.com/patient-graph-lab/ehrshot-patient-graphs.git
    os.chdir(REPO_DIR)

!pip install -e ".[dev,data]"

## STOP — manual step before running the next cell

The next cell prompts you to upload the dataset directly — no Drive, no third-party storage. It does **not** split the large file for you; that has to happen on your own machine first, before you run the cell below:

```bash
python scripts/dismantle_data.py
```

That only applies to `EHRSHOT_ASSETS.zip` (4.3GB) — the other 3 files are small enough to upload as-is, no splitting needed. Writes pieces into `./data-chunks/`.

1. Run the command above, on your own machine, wherever the dataset files live (this needs a local checkout of this repo, or just the one script file).
2. Run the next cell — it'll pop up an upload dialog.
3. In that dialog, **select all of**: `EHRSHOT_MEDS.zip`, `meds_omop_ehrshot.tar.gz`, `meds_reader_omop_ehrshot.tar.gz`, and every file in `data-chunks/` (multi-select — all at once).
4. The pieces get reassembled into `EHRSHOT_ASSETS.zip` automatically.

In [ ]:
from scripts.colab_setup import ensure_ehrshot_data

data_root = ensure_ehrshot_data(checksums_path="data/checksums.txt")

In [ ]:
# Sanity check: confirm the extracted assets look right before doing anything else

assets_dir = data_root / "EHRSHOT_ASSETS"
for sub in ["benchmark", "features", "models", "results", "splits", "data"]:
    p = assets_dir / sub
    print(f"{sub:12s} exists={p.exists()}")

In [ ]:
import pandas as pd

labels = pd.read_csv(assets_dir / "benchmark" / "all_labels.csv")
print(labels.shape)
labels.head()

In [ ]:
import ehrgraph

print("ehrgraph package imports cleanly:", ehrgraph.__name__)

## Verify meds_reader against the real data (issue #1)

This is the one check that can't be done outside Colab — it needs the real, extracted MEDS reader database. Run the cell below and confirm it prints real events with no errors.

In [ ]:
from ehrgraph.data.loader import open_database

db_path = data_root / "meds_reader_omop_ehrshot"
db = open_database(str(db_path))

first_subject_id = next(iter(db))
subject = db[first_subject_id]
print(f"subject_id={subject.subject_id}")
for i, event in enumerate(subject.events):
    if i >= 5:
        break
    print(f"  {event.time}  {event.code}")